In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read data
df_benchmark = pd.read_csv("benchmark.csv")

# Parse semicolon-separated data: 3 metadata columns + 101 histogram bins (0..100)
parsed = df_benchmark.iloc[:, 0].str.rstrip(";").str.split(";", expand=True)

hist_cols = [f"Hist_{i}" for i in range(101)]
parsed = parsed.iloc[:, :3 + len(hist_cols)]
parsed.columns = ["Superpixels", "Compactness", "DiceCoefficient", *hist_cols]

parsed = parsed.apply(pd.to_numeric, errors="coerce")

# Filter out rows where DiceCoefficient is zero
parsed = parsed[parsed["DiceCoefficient"] > 0]

# Average out the DiceCoefficient for each combination of Superpixels and Compactness
parsed = parsed.groupby(["Superpixels", "Compactness"], as_index=False)["DiceCoefficient"].mean()

# Build grid for surface plot
grid = parsed.pivot(index="Compactness", columns="Superpixels", values="DiceCoefficient")
x = grid.columns.to_numpy()
y = grid.index.to_numpy()
X, Y = np.meshgrid(x, y)
Z = grid.to_numpy()

# 2.5D surface plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor="none")

ax.set_xlabel("Superpixels")
ax.set_ylabel("Compactness")
ax.set_zlabel("DiceCoefficient")
ax.view_init(elev=30, azim=225)
#ax.set_title("DiceCoefficient vs Superpixels and Compactness")
fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.1, label="DiceCoefficient")

plt.tight_layout()
plt.show()

In [ ]:
# Average DiceCoefficient over compactness for each superpixel count
mean_by_superpixels = grid.mean(axis=0)

# Average DiceCoefficient over superpixel counts for each compactness value
mean_by_compactness = grid.mean(axis=1)

fig_avg, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(mean_by_superpixels.index, mean_by_superpixels.values, marker="o", linewidth=1.8)
axes[0].set_title("Avg DiceCoefficient vs Superpixels")
axes[0].set_xlabel("Superpixels")
axes[0].set_ylabel("Avg DiceCoefficient")
axes[0].grid(alpha=0.3)

axes[1].plot(mean_by_compactness.index, mean_by_compactness.values, marker="o", linewidth=1.8)
axes[1].set_title("Avg DiceCoefficient vs Compactness")
axes[1].set_xlabel("Compactness")
axes[1].set_ylabel("Avg DiceCoefficient")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Parallel coordinates view of the same DiceCoefficient surface data
pc_df = grid.reset_index().copy()
pc_df["Compactness"] = pc_df["Compactness"].astype(int).astype(str)

ax_pc = pd.plotting.parallel_coordinates(
	pc_df,
	class_column="Compactness",
	colormap="viridis",
	linewidth=1.8,
	alpha=0.9
)

ax_pc.set_title("Dice coeff for multiple compactness and SP counts")
ax_pc.set_xlabel("Superpixels")
ax_pc.set_ylabel("DiceCoefficient")
plt.xticks(rotation=45)
plt.legend(title="Compactness", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Re-parse full benchmark rows to recover histogram bins
full = df_benchmark.iloc[:, 0].str.rstrip(";").str.split(";", expand=True)
full_cols = ["Superpixels", "Compactness", "DiceCoefficient", *hist_cols]
full = full.iloc[:, :len(full_cols)]
full.columns = full_cols
full = full.apply(pd.to_numeric, errors="coerce")

# 
full = full[full["DiceCoefficient"] > 0]

# Average histogram value per bin across all rows
hist_mean = full[hist_cols].mean(axis=0)
bin_idx = np.arange(len(hist_cols))



plt.figure(figsize=(10, 4))
plt.plot(bin_idx, hist_mean.values, marker="o", linewidth=1.5, markersize=3)
plt.fill_between(bin_idx, hist_mean.values, alpha=0.2)
plt.title("Distribution of SP mask overlap with GT mask")
plt.xlabel("Percentage of SP mask in GT mask")
plt.ylabel("SP count")
plt.yscale("log")
plt.xticks(np.arange(0, len(hist_cols), 10))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()